# Daily Weather Pipeline — Walkthrough

This notebook drives the same code the Airflow DAG calls — it **imports**
`extract_load.pipeline`, it does not reimplement any of its logic.

Three things get proven end-to-end below:

1. Extract + load one logical date from Open-Meteo into Postgres, fields unmodified.
2. The load is **idempotent** — running the exact same date twice does not change the row count.
3. The finished dbt mart (`mart_daily_weather`) holds business-ready, city-level daily data, queried with pandas.

In [ ]:
#!pip install pandas psycopg2-binary requests python-dotenv sqlalchemy matplotlib

In [ ]:


import os
import sys
import subprocess
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
import pandas as pd
from sqlalchemy import create_engine
import matplotlib.pyplot as plt

# 1. Load credentials 
load_dotenv(find_dotenv())

# 2. Make the sibling `extract_load` package importable
REPO_ROOT = Path.cwd().parent  # setting directory
sys.path.insert(0, str(REPO_ROOT))

from extract_load.pipeline import run as run_extract_load
from extract_load.db import connection_scope, fetch_row_count

# Pick a date from the past
LOGICAL_DATE = "2024-06-01"

## 1. Extract & load

Calls the Open-Meteo archive API once per city for `LOGICAL_DATE` and upserts into `raw_weather`.

In [ ]:
# Calls the Open-Meteo archive API once per city and upserts into raw_weather.
rows_loaded = run_extract_load(LOGICAL_DATE)

with connection_scope() as conn:
    total_for_date = fetch_row_count(conn, LOGICAL_DATE)

print(f"run() reported {rows_loaded} rows upserted")
print(f"raw_weather now has {total_for_date} rows for {LOGICAL_DATE}")

assert total_for_date == total_for_date_again, "Row count changed on rerun — load is NOT idempotent!"
print("Idempotency holds: row count is unchanged after a duplicate run.")

## 2. Prove idempotency

Run the **exact same logical date again**. If the load is truly idempotent, the row count for
that date must not change — the second run should only update existing rows, never add new ones.

In [ ]:
# Run the exact same logical date again. 
rows_loaded_again = run_extract_load(LOGICAL_DATE)

with connection_scope() as conn:
    total_for_date_again = fetch_row_count(conn, LOGICAL_DATE)

print(f"Second run reported {rows_loaded_again} rows upserted")
print(f"raw_weather now has {total_for_date_again} rows for {LOGICAL_DATE}")

assert total_for_date == total_for_date_again, "Row count changed on rerun — load is NOT idempotent!"
print("Idempotency holds: row count is unchanged after a duplicate run.")

## 3. Run dbt: staging → mart → tests

This is normally the Airflow DAG's job (`dbt run` then `dbt test`). Running it inline here too
so this notebook is a genuine end-to-end proof rather than stopping at the raw table.

In [ ]:
DBT_PROJECT_DIR = REPO_ROOT / "dbt" / "weather_pipeline"

# Added the --profiles-dir flag pointing to the same directory
run_result = subprocess.run(
    ["dbt", "run", "--project-dir", str(DBT_PROJECT_DIR), "--profiles-dir", str(DBT_PROJECT_DIR)],
    capture_output=True, text=True,
)
print(run_result.stdout[-2000:])
if run_result.returncode != 0:
    print(run_result.stderr[-2000:])
run_result.check_returncode()

test_result = subprocess.run(
    ["dbt", "test", "--project-dir", str(DBT_PROJECT_DIR), "--profiles-dir", str(DBT_PROJECT_DIR)],
    capture_output=True, text=True,
)
print(test_result.stdout[-2000:])
if test_result.returncode != 0:
    print(test_result.stderr[-2000:])
test_result.check_returncode()
print("dbt run + dbt test both passed.")

## 4. Query the finished mart

`mart_daily_weather` is the business-ready table: one row per city per day, with a trailing
7-day rolling average max temperature and a hot/mild/cold classification.

In [ ]:
# Replaced PGUSER with POSTGRES_USER to match your .env file
engine = create_engine(
    f"postgresql+psycopg2://{os.environ.get('POSTGRES_USER')}:"
    f"{os.environ.get('POSTGRES_PASSWORD')}@"
    f"{os.environ.get('POSTGRES_HOST')}:"
    f"{os.environ.get('POSTGRES_PORT')}/"
    f"{os.environ.get('POSTGRES_DB')}"
)

mart_df = pd.read_sql(
    "select * from mart_daily_weather order by city, weather_date",
    engine,
)
display(mart_df.head(10))

### Quick look: max temperature by city over the window

In [ ]:
import matplotlib.pyplot as plt

pivot = mart_df.pivot(index="weather_date", columns="city", values="temp_max_c")
pivot.plot(figsize=(10, 5), title="Daily max temperature by city")
plt.ylabel("°C")
plt.show()